# **IMBALANCE ANALYSIS AND MITIGATION**

## **IMPORTS**

In [16]:
# General imports
import pandas as pd
import numpy as np
from imblearn.pipeline import make_pipeline as imb_make_pipeline
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.metrics import make_scorer, cohen_kappa_score

# Preprocessor imports
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from preprocessing.preprocessor import preprocessor

# Oversampling imports
from imblearn.over_sampling import SMOTE, RandomOverSampler, ADASYN, BorderlineSMOTE

# Undersampling imports
from imblearn.under_sampling import RandomUnderSampler, EditedNearestNeighbours, NearMiss, TomekLinks

# Combination imports
from imblearn.combine import SMOTEENN, SMOTETomek

# Model imports
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier, BaggingClassifier
import xgboost as xgb
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

## **DATASET LOADING**

In [17]:
# We read the partitions
X_train = pd.read_parquet("../data/cleaned/X_train.parquet")
X_test = pd.read_parquet("../data/cleaned/X_test.parquet")
y_train = pd.read_parquet("../data/cleaned/y_train.parquet").squeeze()
y_test = pd.read_parquet("../data/cleaned/y_test.parquet").squeeze()

# Check that all has been loaded correctly
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(10045, 32)
(10045,)
(4948, 32)
(4948,)


## **MODELS & TECHNIQUES**

### Models

In [18]:
# MODELS: default hyperparameters, random_state=42
MODELS = {
    'Random Forest': RandomForestClassifier(random_state=42),
    'AdaBoost': AdaBoostClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'Bagging': BaggingClassifier(random_state=42),
    'XGBoost': xgb.XGBClassifier(random_state=42, eval_metric='logloss'),
    'LightGBM': LGBMClassifier(random_state=42, verbose=-1),
    'CatBoost': CatBoostClassifier(random_state=42, verbose=0, allow_writing_files=False)
}

### Balanced models

In [19]:
# MODELS_BALANCED: same models, class_weight='balanced' (where possible)
MODELS_BALANCED = {
    'Random Forest': RandomForestClassifier(random_state=42, class_weight='balanced'),
    'AdaBoost': AdaBoostClassifier(random_state=42),  # doesn't have class_weight
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),  # doesn't have class_weight 
    'Bagging': BaggingClassifier(random_state=42),  # doesn't have class_weight 
    'XGBoost': xgb.XGBClassifier(random_state=42, eval_metric='logloss'),  # doesn't have class_weight
    'LightGBM': LGBMClassifier(random_state=42, verbose=-1, class_weight='balanced'),
    'CatBoost': CatBoostClassifier(random_state=42, verbose=0, auto_class_weights='Balanced', allow_writing_files=False)
}

### Balancing techniques

In [20]:
# BALANCING_TECHNIQUES: techniques to mitigate the class imbalance
BALANCING_TECHNIQUES = {
    'None': 'passthrough',  # to compare without resampling
    'SMOTE (oversampling)': SMOTE(random_state=42),
    'RandomOverSampler (oversampling)': RandomOverSampler(random_state=42),
    'ADASYN (oversampling)': ADASYN(random_state=42, sampling_strategy='minority'),
    'BorderlineSMOTE (oversampling)': BorderlineSMOTE(random_state=42),
    'RandomUnderSampler (undersampling)': RandomUnderSampler(random_state=42),
    'EditedNearestNeighbours (undersampling)': EditedNearestNeighbours(),
    'NearMiss (undersampling)': NearMiss(),
    'TomekLinks (undersampling)': TomekLinks(),
    'SMOTEENN (combination)': SMOTEENN(random_state=42),
    'SMOTETomek (combination)': SMOTETomek(random_state=42)
}

### Initialization

In [21]:
# Initializing the folds that we will be using
skf = StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

# Initializng the second evaluation technique
qwk = make_scorer(cohen_kappa_score, weights="quadratic")
scoring = {
        "f1_macro": "f1_macro",
        "QWK": qwk
    }

# Initializing the results
results = []

## **IMBALANCE ANALYSIS**

In [22]:
target_distribution_train = y_train.value_counts()
target_distribution_test = y_test.value_counts()


for key, amount_train, amount_test in zip(target_distribution_train.index, target_distribution_train.values, target_distribution_test):
    print(f"""Class {key}: 
        Amount train: {amount_train} --> {round((amount_train/y_train.shape[0])*100, 2)}%
        Amount test: {amount_test} --> {round((amount_test/y_test.shape[0])*100, 2)}%""")


Class 4: 
        Amount train: 2812 --> 27.99%
        Amount test: 1385 --> 27.99%
Class 2: 
        Amount train: 2705 --> 26.93%
        Amount test: 1332 --> 26.92%
Class 3: 
        Amount train: 2183 --> 21.73%
        Amount test: 1076 --> 21.75%
Class 1: 
        Amount train: 2070 --> 20.61%
        Amount test: 1020 --> 20.61%
Class 0: 
        Amount train: 275 --> 2.74%
        Amount test: 135 --> 2.73%


## **IMBALANCE MITIGATION**

### Base models

In [23]:
# First we execute the base models

for model_name, model in MODELS_BALANCED.items():
    pipe = imb_make_pipeline(preprocessor, model)
    results_cv = cross_validate(pipe, X_train, y_train, cv=skf, scoring=scoring)

    result = {
        "Model": f"{model_name} balanced",
        "balancing_technique": "no",
        "f1_mean": np.mean(results_cv["test_f1_macro"]),
        "f1_sd": np.std(results_cv["test_f1_macro"]),
        "qwk_mean": np.mean(results_cv["test_QWK"]),
        "qwk_sd": np.std(results_cv["test_QWK"])
    }
    results.append(result)
    print(f"{model_name} -> F1-macro: {round(result['f1_mean'],2)}, QWK: {round(result['qwk_mean'],2)}")


Random Forest -> F1-macro: 0.34, QWK: 0.37
AdaBoost -> F1-macro: 0.28, QWK: 0.3
Gradient Boosting -> F1-macro: 0.35, QWK: 0.38
Bagging -> F1-macro: 0.32, QWK: 0.35
XGBoost -> F1-macro: 0.34, QWK: 0.39


c:\Users\iker\anaconda3\envs\machine_learning\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\iker\anaconda3\envs\machine_learning\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\iker\anaconda3\envs\machine_learning\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\iker\anaconda3\envs\machine_learning\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\iker\anaconda3\envs\machine_learning\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature nam

LightGBM -> F1-macro: 0.34, QWK: 0.37
CatBoost -> F1-macro: 0.35, QWK: 0.38


In [24]:
# Save partial results to a csv
partial_results = pd.DataFrame(results)

partial_results.to_csv("technique_results/partial_results.csv")



### Models & balancing techniques

In [ ]:
# We check all the combinations of balancing techniques and models
#! DON'T EXECUTE AGAIN (+1 hour)

for technique_name, technique in BALANCING_TECHNIQUES.items():
    for model_name, model in MODELS.items():
        pipe = imb_make_pipeline(preprocessor, technique, model)
        
        results_cv = cross_validate(pipe, X_train, y_train, cv=skf, scoring=scoring)

        result = {
            "Model": model_name,
            "balancing_technique": technique_name,
            "f1_mean": np.mean(results_cv["test_f1_macro"]),
            "f1_sd": np.std(results_cv["test_f1_macro"]),
            "qwk_mean": np.mean(results_cv["test_QWK"]),
            "qwk_sd": np.std(results_cv["test_QWK"])
        }

        results.append(result)

        print(f"{model_name} using {technique_name} -> F1-macro: {round(result['f1_mean'],2)}, QWK: {round(result['qwk_mean'],2)}")




Random Forest using None -> F1-macro: 0.34, QWK: 0.36
AdaBoost using None -> F1-macro: 0.28, QWK: 0.3
Gradient Boosting using None -> F1-macro: 0.35, QWK: 0.38
Bagging using None -> F1-macro: 0.32, QWK: 0.35
XGBoost using None -> F1-macro: 0.34, QWK: 0.39


c:\Users\iker\anaconda3\envs\machine_learning\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\iker\anaconda3\envs\machine_learning\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\iker\anaconda3\envs\machine_learning\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\iker\anaconda3\envs\machine_learning\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\iker\anaconda3\envs\machine_learning\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature nam

LightGBM using None -> F1-macro: 0.33, QWK: 0.38
CatBoost using None -> F1-macro: 0.32, QWK: 0.39
Random Forest using SMOTE (oversampling) -> F1-macro: 0.36, QWK: 0.38
AdaBoost using SMOTE (oversampling) -> F1-macro: 0.29, QWK: 0.26
Gradient Boosting using SMOTE (oversampling) -> F1-macro: 0.35, QWK: 0.37
Bagging using SMOTE (oversampling) -> F1-macro: 0.33, QWK: 0.33
XGBoost using SMOTE (oversampling) -> F1-macro: 0.35, QWK: 0.4


c:\Users\iker\anaconda3\envs\machine_learning\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\iker\anaconda3\envs\machine_learning\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\iker\anaconda3\envs\machine_learning\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\iker\anaconda3\envs\machine_learning\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\iker\anaconda3\envs\machine_learning\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature nam

LightGBM using SMOTE (oversampling) -> F1-macro: 0.34, QWK: 0.37
CatBoost using SMOTE (oversampling) -> F1-macro: 0.34, QWK: 0.39
Random Forest using RandomOverSampler (oversampling) -> F1-macro: 0.36, QWK: 0.39
AdaBoost using RandomOverSampler (oversampling) -> F1-macro: 0.28, QWK: 0.25
Gradient Boosting using RandomOverSampler (oversampling) -> F1-macro: 0.33, QWK: 0.34
Bagging using RandomOverSampler (oversampling) -> F1-macro: 0.32, QWK: 0.32
XGBoost using RandomOverSampler (oversampling) -> F1-macro: 0.36, QWK: 0.39


...

LightGBM using TomekLinks (undersampling) -> F1-macro: 0.33, QWK: 0.37
CatBoost using TomekLinks (undersampling) -> F1-macro: 0.31, QWK: 0.38
Random Forest using SMOTEENN (combination) -> F1-macro: 0.17, QWK: 0.08
AdaBoost using SMOTEENN (combination) -> F1-macro: 0.12, QWK: 0.09
Gradient Boosting using SMOTEENN (combination) -> F1-macro: 0.18, QWK: 0.09
Bagging using SMOTEENN (combination) -> F1-macro: 0.18, QWK: 0.1
XGBoost using SMOTEENN (combination) -> F1-macro: 0.22, QWK: 0.17


c:\Users\iker\anaconda3\envs\machine_learning\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\iker\anaconda3\envs\machine_learning\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\iker\anaconda3\envs\machine_learning\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\iker\anaconda3\envs\machine_learning\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\iker\anaconda3\envs\machine_learning\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature nam

LightGBM using SMOTEENN (combination) -> F1-macro: 0.22, QWK: 0.16
CatBoost using SMOTEENN (combination) -> F1-macro: 0.19, QWK: 0.11
Random Forest using SMOTETomek (combination) -> F1-macro: 0.35, QWK: 0.37
AdaBoost using SMOTETomek (combination) -> F1-macro: 0.28, QWK: 0.26
Gradient Boosting using SMOTETomek (combination) -> F1-macro: 0.34, QWK: 0.37
Bagging using SMOTETomek (combination) -> F1-macro: 0.33, QWK: 0.34
XGBoost using SMOTETomek (combination) -> F1-macro: 0.35, QWK: 0.39


c:\Users\iker\anaconda3\envs\machine_learning\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\iker\anaconda3\envs\machine_learning\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\iker\anaconda3\envs\machine_learning\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\iker\anaconda3\envs\machine_learning\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\iker\anaconda3\envs\machine_learning\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature nam

LightGBM using SMOTETomek (combination) -> F1-macro: 0.33, QWK: 0.37
CatBoost using SMOTETomek (combination) -> F1-macro: 0.34, QWK: 0.39


In [26]:
# Save results to a csv
final_results = pd.DataFrame(results)
final_results.to_csv("technique_results/final_results.csv")
